# agent-serving-sim quickstart

Generate a synthetic agent workload, replay it under different KV cache eviction
policies, and compare JCT / hit-rate. Requires `pip install -e ".[dev,viz]"`.

In [ ]:
from ass.cache.policies import LRUPolicy, TTLPolicy
from ass.scheduler.serving import ServingConfig, ServingSim
from ass.workload.synthetic import AgentProfile, SyntheticConfig, generate_trace
from ass.workload.schema import write_trace

# 1) Heterogeneous load: fast-cycling coding agents + slow search agents
config = SyntheticConfig(
    num_sessions=150,
    turns_per_session=8,
    session_arrival_rate=0.3,
    agent_mix={"coding": 100, "search": 50},
    agent_profiles={
        "coding": AgentProfile(think_time_mu=1.61, think_time_sigma=0.5),   # median ~5s
        "search": AgentProfile(think_time_mu=3.40, think_time_sigma=0.7),   # median ~30s
    },
)
trace = generate_trace(config, seed=42)
print(f"{len(trace)} requests, {config.num_sessions} sessions")
trace[:2]

In [ ]:
# 2) Replay under LRU vs several TTLs on the same trace
serving = ServingConfig(cache_capacity_tokens=60_000, decode_chunks=4)

def run(policy):
    sim = ServingSim(serving, policy=policy)
    sim.submit_all(trace)
    sim.run()
    return sim.collector

runs = {"lru": run(LRUPolicy())}
for ttl in (5, 15, 30, 120):
    runs[f"ttl-{ttl}"] = run(TTLPolicy(ttl=ttl))

for label, collector in runs.items():
    s = collector.summary()
    print(
        f"{label:>8}: hit={s['hit_rate']:.3f}  jct_mean={s['jct_mean']:.2f}s  "
        f"p95={s['jct_p95']:.2f}s  evictions={s['evictions']['count']}"
    )

In [ ]:
# 3) JCT CDF per class for the knee TTL vs LRU
from ass.viz.plots import plot_cdf

plot_cdf(
    {
        "lru (coding)": runs["lru"].jct_values(agent_type="coding"),
        "ttl-15 (coding)": runs["ttl-15"].jct_values(agent_type="coding"),
        "lru (search)": runs["lru"].jct_values(agent_type="search"),
        "ttl-15 (search)": runs["ttl-15"].jct_values(agent_type="search"),
    },
    "jct_cdf_quickstart.png",
    title="JCT CDF by agent type",
    xlabel="JCT (s)",
)
print("figure written to jct_cdf_quickstart.png")

## Next steps

- Replay the **bundled real trace** (`traces/real/*.jsonl`, 1021 requests) with
  `experiments/exp005_real_trace_replay.py`;
- Try preemption economics: `ServingConfig(evict_tps=2000, decode_chunks=4)` —
  see `experiments/exp006_preemption_and_eviction_cost.py`;
- Implement your own eviction policy: subclass `EvictionPolicy`, decorate with
  `register_policy`, instantiate via `create_policy(name, **kwargs)` — no kernel
  changes needed.